# 01 BIDMC Label Readiness Audit

Audit SpO2 quality and low-oxygen coverage before canonical export.

Inputs from: `outputs/bidmc_dataset_file_coverage.csv` (from notebook 00).
Outputs to: `outputs/bidmc_audit.csv`.
Next: notebook 02 for canonical window export.

## SpO2 Quality Assessment Setup

Load the eligible recording list from notebook 00 and establish SpO2 thresholds for low-oxygen and borderline-range diagnostics.

In [ ]:
from pathlib import Path
import os
import pandas as pd

LOW_SPO2_THRESHOLD = 90.0
BORDERLINE_LOW = 89.0
BORDERLINE_HIGH = 91.0

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'notebooks').is_dir() and (candidate / 'outputs').is_dir():
            return candidate
    raise RuntimeError('Could not find project root containing notebooks/ and outputs/')

def resolve_bidmc_dir(project_root: Path) -> Path:
    env_path = os.getenv('BIDMC_DATASET_DIR', '').strip()
    if env_path:
        p = Path(env_path).expanduser().resolve()
        if p.is_dir():
            return p
    default_path = (project_root / 'bidmc_csv').resolve()
    if default_path.is_dir():
        return default_path
    raise FileNotFoundError('BIDMC dataset directory not found. Set BIDMC_DATASET_DIR or place data at <repo>/bidmc_csv')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
OUTPUTS_DIR = PROJECT_ROOT / 'outputs'
COVERAGE_CSV = OUTPUTS_DIR / 'bidmc_dataset_file_coverage.csv'
AUDIT_CSV = OUTPUTS_DIR / 'bidmc_audit.csv'
DATASET_DIR = resolve_bidmc_dir(PROJECT_ROOT)

if not COVERAGE_CSV.is_file():
    raise FileNotFoundError(f'Missing {COVERAGE_CSV}. Run notebook 00 first.')
coverage_df = pd.read_csv(COVERAGE_CSV)
eligible_df = coverage_df.loc[coverage_df['all_required_present']].copy()
print(f'Eligible recordings: {len(eligible_df)}')


## Low-Oxygen Coverage Analysis

For each eligible recording, compute missingness, low-SpO2 counts, borderline counts, and min/max SpO2 to produce a readiness report.

In [ ]:
rows = []
for row in eligible_df.itertuples(index=False):
    signal_df = pd.read_csv(Path(row.signals_path))
    numerics_df = pd.read_csv(Path(row.numerics_path))

    spo2_cols = [c for c in numerics_df.columns if c.strip().lower() == 'spo2']
    if not spo2_cols:
        raise KeyError(f'SpO2 column not found for {row.recording_id}')
    spo2 = pd.to_numeric(numerics_df[spo2_cols[0]], errors='coerce')
    spo2_valid = spo2.dropna()

    rows.append({
        'recording_id': row.recording_id,
        'signal_row_count': int(len(signal_df)),
        'numerics_row_count': int(len(numerics_df)),
        'spo2_missing_count': int(spo2.isna().sum()),
        'spo2_below_90_count': int((spo2_valid < LOW_SPO2_THRESHOLD).sum()),
        'spo2_89_to_91_count': int(((spo2_valid >= BORDERLINE_LOW) & (spo2_valid <= BORDERLINE_HIGH)).sum()),
        'spo2_min': float(spo2_valid.min()) if not spo2_valid.empty else float('nan'),
        'spo2_max': float(spo2_valid.max()) if not spo2_valid.empty else float('nan'),
    })

audit_df = pd.DataFrame(rows).sort_values('recording_id').reset_index(drop=True)
audit_df.to_csv(AUDIT_CSV, index=False)
print(f'Wrote: {AUDIT_CSV}')
audit_df.head()
